In [ ]:
#exhaustive search for CEDRUS+ parameters

import collections
import csv

tsec = 256
maxsigs = 2^64
if tsec == 128:
    maxsigbytes = 18000
    maxcalls = 4400000
if tsec == 192:
    maxsigbytes = 36000
    maxcalls = 7500000
if tsec == 256:
    maxsigbytes = 50000
    maxcalls = 6600000

class memoized(object):
  def __init__(self,func):
    self.func = func
    self.cache = {}
    self.__name__ = 'memoized:' + func.__name__
  def __call__(self,*args):
    if not isinstance(args,collections.abc.Hashable):
      return self.func(*args)
    if not args in self.cache:
      self.cache[args] = self.func(*args)
    return self.cache[args]

F=RealField(tsec+100)


@memoized
def qhitprob(leaves, qs, r):
    p = F(1) / leaves
    return binomial(qs, r) * (p**r) * ((1-p)**(qs-r))

@memoized
def forgeryprob(b, r, k):
    base_prob = 1 - (1 - F(1)/F(2**b))**r
    return base_prob**k

def compute_mincost(h,d):
    y=h%d
    if y == 0:
        h_ary = [int(h/d)]*d
    else:
        h1 = int(h/d)
        h2 = ceil(h/d)
        k = h-h1*d
        h_ary = [h1]*(d-k) + [h2]*k
    sum_h = 0
    for i in h_ary:
        sum_h += 2 ** i
    return sum_h

def compute_min_w_ary(A,l):
    y=A%l
    if y == 0:
        w_ary = [2**int(A/l)]*l
    else:
        W1 = int(A/l)
        W2 = ceil(A/l)
        k = A-W1*l
        w_ary = [2**W1]*(l-k) + [2**W2]*k
    return w_ary

def run_script():
    hashbytes = ceil(tsec/8)
    para_list = [0]*(4*hashbytes+10)
    for l0 in range(hashbytes,4*hashbytes+1):
        w_ary1 = compute_min_w_ary(tsec, l0)
        l0_bit = int(log(sum(w_ary1)-l0,2)) + 1
        for l1 in range(ceil(l0_bit/8),int(l0_bit/2)+1):
            w_ary2 = compute_min_w_ary(l0_bit, l1)
            para_list[len(w_ary1+w_ary2)] = w_ary1+w_ary2
    para_list = list(filter(lambda x: x != 0, para_list))
    with open("CEDRUS+_256.csv","w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['h', 'd', 'b', 'k', 'w_ary', 'l0+l1', 'Size', 'sig_speed', 'vrfy_speed','sec'])
        sigmalimit = F(2**(-tsec))
        donelimit = 1-sigmalimit/2**(20)
        for h in range(58, 69):
            leaves = 2**h
            for b in range(3,24):
                for k in range(1,64):
                    sigma = 0
                    r = 1
                    done = qhitprob(leaves, maxsigs, 0)
                    while done < donelimit:
                        t =qhitprob(leaves, maxsigs, r)
                        sigma += t*forgeryprob(b, r, k)
                        if sigma > sigmalimit:
                            break
                        done += t
                        r += 1
                    sigma += min(0, 1-done)
                    if sigma > sigmalimit:
                        continue
                    for d in range(3,h):
                        print(h,b,k,d)
                        for w_ary in para_list:
                            sig_size = ((b + 1) * k + h + 1 + len(w_ary)*d ) * hashbytes
                            if sig_size < maxsigbytes:
                                sign_speed = 3*k*2**b-k+1 + compute_mincost(h, d) * (sum(w_ary)+2)-d
                                if sign_speed < maxcalls:
                                    vrfy_speed = k * (b + 1) + h + d * (sum(w_ary)-len(w_ary)+1)+1
                                    sec = -log(sigma, 2)
                                    writer.writerow([h,d,b,k,w_ary,len(w_ary),sig_size,float(sign_speed),vrfy_speed,sec])
                    break
run_script()